# SkillOutcome SIH26135 ML workflow

This notebook inspects normalized snapshots and reproducible model-comparison metrics. Run data preparation and training first.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'data' / 'processed' / 'placement_training_snapshot.csv'
REPORTS = ROOT / 'reports'
df = pd.read_csv(DATA)
df.shape, df['Trainee_ID'].nunique(), df[['Trainee_ID', 'Target_Job_Role', 'Skill_Gap_Score']].head()

FileNotFoundError: [Errno 2] No such file or directory: '/data/processed/placement_training_snapshot.csv'

In [2]:
cohort_summary = df[['Placement_Target', 'Attendance_Percent', 'Assessment_Score', 'Training_Performance', 'Skill_Gap_Score', 'Demand_Score']].describe().T
cohort_summary

,count,mean,std,min,25%,50%,75%,max
Placement_Target,50000.0,0.861700,0.345218,0.00,1.00,1.000,1.00,1.00
Attendance_Percent,50000.0,77.737286,12.504360,35.00,69.30,78.000,86.70,100.00
Assessment_Score,50000.0,72.794804,13.054976,25.00,63.90,73.000,81.90,100.00
Training_Performance,50000.0,75.020200,10.837313,31.71,67.65,75.245,82.73,100.00
Skill_Gap_Score,50000.0,87.203029,17.468408,0.00,73.86,100.000,100.00,100.00
Demand_Score,50000.0,42.240626,13.730115,13.64,32.11,41.770,51.65,76.64


In [3]:
for name in ['placement_model_metrics.json', 'attrition_model_metrics.json']:
    metrics = json.loads((REPORTS / name).read_text(encoding='utf-8'))
    print(name, metrics['selected_algorithm'], metrics['test_metrics'])

placement_model_metrics.json logistic_regression {'threshold': 0.5, 'roc_auc': 0.8145, 'average_precision': 0.9596, 'positive_rate_baseline_average_precision': 0.856, 'accuracy': 0.8677, 'precision': 0.8797, 'recall': 0.9794, 'f1': 0.9269, 'brier_score': 0.0989, 'confusion_matrix': [[226, 883], [136, 6457]]}
attrition_model_metrics.json xgboost {'threshold': 0.5, 'roc_auc': 0.7148, 'average_precision': 0.1407, 'positive_rate_baseline_average_precision': 0.0643, 'accuracy': 0.9357, 'precision': 0.5, 'recall': 0.0044, 'f1': 0.0086, 'brier_score': 0.0583, 'confusion_matrix': [[6678, 2], [457, 2]]}


## Model comparison

Every candidate is scored on the chronological validation split; the selected algorithm is the winner on the task's selection metric.

In [4]:
comparison = {}
for name in ['placement_model_metrics.json', 'attrition_model_metrics.json']:
    metrics = json.loads((REPORTS / name).read_text(encoding='utf-8'))
    comparison[metrics['model_name']] = pd.DataFrame(metrics['model_comparison']).T.assign(selected=lambda frame: frame.index == metrics['selected_algorithm'])
comparison['placement_model']

,roc_auc,average_precision,brier_score,selected
logistic_regression,0.8080,0.9597,0.0948,True
logistic_regression_balanced,0.8080,0.9597,0.1766,False
random_forest,0.7984,0.9572,0.0965,False
random_forest_balanced,0.7981,0.9576,0.1073,False
xgboost,0.8026,0.9590,0.0960,False
xgboost_weighted,0.8026,0.9590,0.0960,False


In [5]:
comparison['attrition_model']

,roc_auc,average_precision,brier_score,selected
logistic_regression,0.7219,0.1734,0.0599,False
logistic_regression_balanced,0.7195,0.1715,0.2078,False
random_forest,0.6986,0.1588,0.0606,False
random_forest_balanced,0.7137,0.1581,0.0677,False
xgboost,0.7148,0.1756,0.0601,True
xgboost_weighted,0.7059,0.1565,0.1633,False


## Attrition risk bands

Termination is the rare class, so accuracy at 0.5 is uninformative. Read average precision and the metrics at the production thresholds.

In [6]:
attrition_metrics = json.loads((REPORTS / 'attrition_model_metrics.json').read_text(encoding='utf-8'))
print('label:', attrition_metrics['target_description'])
print('thresholds:', attrition_metrics['risk_thresholds'])
print('risk bands on test:', attrition_metrics.get('risk_band_distribution_test'))
pd.DataFrame({
    'at_0.5': attrition_metrics['test_metrics'],
    'at_medium': attrition_metrics.get('test_metrics_at_medium_risk_threshold', {}),
    'at_high': attrition_metrics.get('test_metrics_at_high_risk_threshold', {}),
}).drop(index=['confusion_matrix'], errors='ignore')

label: Termination before the 6-month retention milestone
thresholds: {'medium': 0.0502, 'high': 0.1003, 'selection': 'validation_f1'}
risk bands on test: {'low': 4310, 'medium': 1570, 'high': 1259}


,at_0.5,at_medium,at_high
threshold,0.5,0.0502,0.1003
roc_auc,0.7148,0.7148,0.7148
average_precision,0.1407,0.1407,0.1407
positive_rate_baseline_average_precision,0.0643,0.0643,0.0643
accuracy,0.9357,0.6302,0.8087
precision,0.5,0.1145,0.1398
recall,0.0044,0.7059,0.3834
f1,0.0086,0.1971,0.2049
brier_score,0.0583,0.0583,0.0583


## Data quality and leakage checks

The preparation run records the label definition, the observation windows, and the checks it enforced before committing the tables.

In [7]:
quality = json.loads((REPORTS / 'data_quality.json').read_text(encoding='utf-8'))
print(json.dumps({k: quality[k] for k in ['rows', 'labels', 'snapshot_windows', 'feature_completeness', 'checks_passed']}, indent=2))

{
  "rows": {
    "trainees": 50000,
    "training_completions": 50000,
    "trainee_skills": 150000,
    "role_skill_requirements": 36,
    "skill_gap_details": 150000,
    "job_demand_snapshots": 17664,
    "job_search_events": 50000,
    "employment_spells": 43085,
    "salary_history": 81025,
    "engagement_checkins": 164978,
    "wage_progression_outcomes": 43085,
    "placement_training_snapshot": 50000,
    "attrition_training_snapshot": 40935
  },
  "labels": {
    "placement_positive_rate": 0.8617,
    "attrition_positive_rate": 0.071,
    "attrition_label_definition": "Termination before the 6-month retention milestone, observed strictly after the snapshot checkpoint.",
    "attrition_checkpoint_days": [
      30,
      60,
      90,
      120,
      150
    ],
    "employment_spells_outside_risk_window": 2150
  },
  "snapshot_windows": {
    "placement": [
      "2024-01-01",
      "2025-12-01"
    ],
    "attrition": [
      "2024-02-14",
      "2026-08-10"
    ]
  },
  "f

## Interpretation guardrail

Placement inputs exclude post-outcome columns. The normalized timeline is deterministic synthetic derivation because the supplied example has no actual employment-event history; replace it with operational extracts before production use.